# Gate R95 — improdutividade, produtividade e coverage

Compara a R1 em produção com duas mudanças simples: (1) uma alegação de improdutividade abstém quando o evento já está em dúvida ou tem menos de quatro amostras observadas; (2) ação ainda sem nome volta a ser produtiva somente quando há mãos na máquina e a pessoa já foi identificada como operador. Presença não é recalculada nem alterada.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

ROOT = Path.cwd()
NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
REPO = NB_DIR.parent
sys.path.insert(0, str(NB_DIR))
sys.path.insert(0, str(REPO))

from produtividade_30d import A, I, P, aplicar_regras, carregar_fontes, dividir_por_dia, metricas, normalizar_serie, preparar_dataset, serie_bool

OUT = Path(os.environ.get('KV_R95_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_precision_95'))
OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, fontes = carregar_fontes()
proxy, _ = preparar_dataset(eventos, catalogo)
split = dividir_por_dia(proxy)
print('Fontes:', *(str(p) for p in fontes), sep='\n- ')

In [ ]:
def politicas_r95(df):
    r1 = aplicar_regras(df)['R1_indefinida_abstem'].copy()
    sem_nome = df['_label_pred'].isin({'acao_indefinida', 'nao_nomeado'})
    operador = normalizar_serie(df['papel_pessoa']).eq('operador')
    maos = serie_bool(df['maos_maquina'])
    poucas_amostras = pd.to_numeric(df['n_amostras'], errors='coerce').lt(4)
    veto_i = serie_bool(df['em_duvida']) | poucas_amostras

    r8 = r1.copy()
    r8[sem_nome & operador & maos] = P

    r95 = r1.copy()
    r95[(r95 == I) & veto_i] = A

    combinado = r8.copy()
    combinado[(combinado == I) & veto_i] = A
    return {
        'R1_producao': r1,
        'R8_maos_recuperam_coverage': r8,
        'R95_veto_I_duvida_ou_poucas_amostras': r95,
        'R95_combinado': combinado,
    }

def resumo(parte, politica, df, pred):
    m = metricas(df, pred)
    return {
        'parte': parte,
        'politica': politica,
        'precisao_improdutividade_pct': round(100 * m['precision_I'], 2),
        'precisao_produtividade_pct': round(100 * m['precision_P'], 2),
        'coverage_pct': round(100 * m['coverage'], 2),
        'alegacoes_improdutividade': int(m['claims_I']),
    }

linhas = []
for parte, df in [('calibracao', split.calibracao), ('teste_interno', split.teste_interno)]:
    for nome, pred in politicas_r95(df).items():
        linhas.append(resumo(parte, nome, df, pred))
resultado = pd.DataFrame(linhas)
resultado

In [ ]:
resultado['passa_95_95_65'] = (
    (resultado.precisao_improdutividade_pct >= 95)
    & (resultado.precisao_produtividade_pct >= 95)
    & (resultado.coverage_pct >= 65)
)
resultado['coverage_ideal'] = resultado.coverage_pct >= 75
escolhida = resultado[resultado.politica == 'R95_combinado'].copy()
assert escolhida.passa_95_95_65.all(), resultado
assert escolhida.coverage_ideal.all(), resultado

decisao = {
    'regra_escolhida': 'R95_combinado',
    'gate': {
        'precisao_I_min_pct': 95,
        'precisao_P_min_pct': 95,
        'coverage_min_pct': 65,
        'coverage_ideal_pct': 75,
    },
    'presenca': 'inalterada por construcao; o gate recebe papel_pessoa pronto e muda somente P/I/ABSTEM',
    'resultados': escolhida.to_dict(orient='records'),
}
resultado.to_csv(OUT / 'resultado_gate_r95.csv', index=False)
(OUT / 'decisao_gate_r95.json').write_text(json.dumps(decisao, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(decisao, indent=2, ensure_ascii=False))
resultado